# **Notebook 03: Logistic Regression From Scratch**

先用 TF-IDF 准备好数据

In [310]:
import pandas as pd
from pathlib import Path
import numpy as np

data_path=Path("../data/SMSSpamCollection.txt")
df=pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label","message"]
)

df["label_num"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

from sklearn.model_selection import train_test_split

y = df["label_num"]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    y,
    test_size=0.2,
    random_state=42
)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(X_train.shape)

(4457, 7702)


## 1. 建立模型

在 Logistic Regression 中，我们有这样的公式：

$$ z=\theta ^\top x $$

$$h_{\theta}(x)=\sigma(z)$$

其中，sigmoid 为

$$\sigma(z)=\frac{1}{1+\mathrm{e}^{-z}}$$


In [311]:
### 定义模型参数 θ
n_features = X_train.shape[1]
theta = np.zeros(n_features)

首先实现 sigmoid

In [312]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

然后实现 forward，即 $z=\theta ^\top x $，在 `Numpy` 里用 $ z=X \theta $ 实现

In [313]:
def forward(X, theta):

    z = X @ theta
    h = sigmoid(z)

    return h

## 2. 计算Loss函数

结合 $Bernoulli$：

$$P(y|x;\theta)=h_{\theta}(x)^y (1−h_{\theta}(x))^{1−y}$$

然后 $Likelihood$：

$$L(\theta)=\prod_{i}^{}​ P(y^{(i)}|x^{(i)})$$

$Log-Likelihood$：

$$\sum_{i}^{} [ y\log{h}+(1−y)\log{(1−h)} ]$$

最后为了变成最小化问题，取负号得到 $Cross Entropy$：

$$J(\theta)=−\frac{1}{m} \sum [ y\log{h}+(1−y)\log{(1−h)} ]$$

理论上代码应该是：

In [314]:
# def compute_loss(h, y):

#     m = len(y)

#     loss = -(1/m) * np.sum(
#         y * np.log(h) + (1-y) * np.log(1-h) )

#     return loss

然而，当模型预测 $1$ 或 $0$ 时，计算 $J(\theta)$ 时，会碰到类似于 $\log{0}$ 这种数学上没有良好定义的对象。因此我们会加上：

In [315]:
# epsilon = 1e-15

# h = np.clip(
#     h,
#     epsilon,
#     1-epsilon
# )

这一段，让 $1$ 变成 $0.999999999999999$，让 $0$ 变成 $0.000000000000001$，保障了数学对象的良好定义。因此，完整地，

In [316]:
def compute_loss(h, y):

    epsilon = 1e-15
    h = np.clip(
        h,
        epsilon,
        1-epsilon
    )

    m = len(y)
    loss = -(1/m) * np.sum(
        y*np.log(h)
        +
        (1-y)*np.log(1-h)
    )

    return loss

## 3. Gradient Decent

求函数 $J(\theta)$ 的梯度：

$$\nabla J(\theta)=\frac{1}{m} ​X^\top (h−y)​$$

In [317]:
def compute_gradient(X,h,y):

    m = X.shape[0]
    gradient = (1/m)*X.T@(h-y)

    return gradient

进行一次梯度下降过程 应该包括通过计算现有的 $h_{\theta}$ 和梯度，并用他们来更新 $\theta$，如下

In [318]:
# h = forward(X_train, theta)

# gradient = compute_gradient(
#     X_train,
#     h,
#     y_train
# )

# theta = theta - learning_rate * gradient

所以我们写一个循环，实现梯度下降的学习过程

In [319]:
loss_history = []
learning_rate = 1                  # 学习率
iterations = 10000                 # 优化次数的最大值

for i in range(iterations):

    # Forward
    h = forward(X_train, theta)

    # Loss，即J(θ)
    loss = compute_loss(h, y_train)
    loss_history.append(loss)

    # Gradient
    gradient = compute_gradient(
        X_train,
        h,
        y_train
    )

    # Update
    theta = theta - learning_rate * gradient

    if i % 500 == 0:
        print(
            f"Iteration {i}, Loss: {loss}"
        )

print(
    f"Iteration {i}, Loss: {loss}"
)

Iteration 0, Loss: 0.6931471805599453
Iteration 500, Loss: 0.30188752818679254
Iteration 1000, Loss: 0.22441738299402114
Iteration 1500, Loss: 0.18489887202479238
Iteration 2000, Loss: 0.16015934130501222
Iteration 2500, Loss: 0.14287395590975555
Iteration 3000, Loss: 0.1299207151679941
Iteration 3500, Loss: 0.11973450523106727
Iteration 4000, Loss: 0.1114390243663452
Iteration 4500, Loss: 0.10450237285236952
Iteration 5000, Loss: 0.09858121242900844
Iteration 5500, Loss: 0.09344294154704663
Iteration 6000, Loss: 0.08892371993043459
Iteration 6500, Loss: 0.08490438558295237
Iteration 7000, Loss: 0.08129591912606365
Iteration 7500, Loss: 0.07803029149590086
Iteration 8000, Loss: 0.07505449098576315
Iteration 8500, Loss: 0.07232650348373329
Iteration 9000, Loss: 0.06981253409975882
Iteration 9500, Loss: 0.06748504146644625
Iteration 9999, Loss: 0.06532549518608868


## 4. 测试模型

不断增加 `iterations` 确实可以使得 `Loss` 逐渐变小，但是 `Loss` 减小不一定会使得 `Accuracy` 减小，所以为了防止 **过拟合 (Over Fitting)**，需要及时用测试集检查 `Accuracy`。

In [320]:
### 用当前模型预测测试集

def predict(X, theta):

    h = forward(X, theta)
    y_pred = (h >= 0.5).astype(int)

    return y_pred

y_pred = predict(
    X_test,
    theta
)

### 计算 Accuracy
accuracy = np.mean(
    y_pred == y_test
)

print(
    "Accuracy:",
    accuracy
)

Accuracy: 0.9820627802690582


此外，Confusion Matrix 和 F1 也是评价该模型训练效果的重要条件

In [321]:
from sklearn.metrics import confusion_matrix

def safe_divide(a,b):   # 确保分母不为0
    if b==0:
        return 0
    return a/b

def compute_metrics(y_true, y_pred):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = cm.ravel()
    print("True Negative:", tn)
    print("False Positive:", fp)
    print("False Negative:", fn)
    print("True Positive:", tp)

    precision = safe_divide(tp , (tp + fp))
    recall = safe_divide(tp , (tp + fn))

    f1 = 2 * safe_divide(( precision * recall ) , ( precision + recall ))

    return precision, recall, f1

precision, recall, f1 = compute_metrics(
    y_test,
    y_pred
)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

True Negative: 957
False Positive: 9
False Negative: 11
True Positive: 138
Precision: 0.9387755102040817
Recall: 0.9261744966442953
F1-score: 0.9324324324324326


## 5. 不断对比实验

通过多次测试，发现以下结果

| 实验 | 学习率 α | Irerations | 最终 Loss | Accuracy | F1-score |
|------|------|------|------|------|------|
| 1 | 0.01 | 1000 | 0.665 | 0.870 | 0.099 |
| 2 | 0.01 | 5000 | 0.576 | 0.872 | 0.134 |
| 3 | 0.01 | 10000 | 0.502 | 0.875 | 0.167 |
| 4 | 0.10 | 1000 | 0.502 | 0.875 | 0.167 |
| 5 | 0.10 | 5000 | 0.302 | 0.926 | 0.627 |
| 6 | 0.10 | 10000 | 0.224 | 0.952 | 0.791 |
| 7 | 0.10 | 15000 | 0.185 | 0.963 | 0.848 |
| 8 | 0.10 | 20000 | 0.160 | 0.971 | 0.885 |
| 9 | 1.00 | 5000 | 0.099 | 0.978 | 0.913 |
| 10 | 1.00 | 10000 | 0.065 | 0.982 | 0.932 |

这表明：
1. Learning Rate 会极大地影响收敛速度
2. 增大 irerations 的收益逐步降低
3. Loss 和 Accuracy 基本一致下降